In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


## Baseline results (for reference)

All train & all test
 - Test F1 Score:  0.6338
 - Test Precision: 0.6512
 - Test Recall:    0.6393
 - Test Accuracy:  0.6393

Masking train & all test
 - Test F1 Score:  0.5921
 - Test Precision: 0.6287
 - Test Recall:    0.6066
 - Test Accuracy:  0.6066

All train & Masking test
 - Test F1 Score:  0.6847
 - Test Precision: 0.7844
 - Test Recall:    0.7049
 - Test Accuracy:  0.7049

Masking train & Masking test
 - Test F1 Score:  0.6304
 - Test Precision: 0.6581
 - Test Recall:    0.6393
 - Test Accuracy:  0.6393

## Improvements
All train & all test
 - Test F1 Score:  0.6539
 - Test Precision: 0.6610
 - Test Recall:    0.6557
 - Test Accuracy:  0.6557

Masking train & all test
 - Test F1 Score:  0.5902
 - Test Precision: 0.5905
 - Test Recall:    0.5902
 - Test Accuracy:  0.5902

All train & Masking test
 - Test F1 Score:  0.6711
 - Test Precision: 0.6758
 - Test Recall:    0.6721
 - Test Accuracy:  0.6721

Masking train & Masking test
 - Test F1 Score:  0.6850
 - Test Precision: 0.7002
 - Test Recall:    0.6885
 - Test Accuracy:  0.6885

# Training Walkthrough

Same staged pipeline as `train_3d_vit()` in `train_model.py`.

1. Configure the experiment (all params match `config.py` defaults).
2. SSL pretraining on BraTS tensors.
3. Multimodal survival training and testing.

In [3]:
from maikol_utils.print_utils import print_separator
from src.config import Configuration
from src.training import train_stage_ssl, train_stage_survival, test_model
from src.utils import set_all_seeds
from scripts import prepare_data

CONFIG = Configuration(
    ssl_epochs=30,
    survival_epochs=20,
    masked_train=True,
    masked_test=True,
    pos_embed="1d",
    use_radiomics=False,
    dynamic_dropout=False,
    ssl_dataset="brats",
    create_folders=False,
)

set_all_seeds(CONFIG.seed)


Configuration initialized
 - seed: 42
 - masked_train: True
 - masked_test: True
 - dataset: brats
 - epochs: 30 (SSL), 20 (Survival)
 - use_radiomics: False
 - dynamic_dropout: False
 - batch_size: 32 (SSL), 16 (Survival)


# Prepare data

In [4]:
print_separator("Preparing Data", sep_type='LONG')
ssl_train_loader, survival_dm, train_loader, val_loader, test_loader = prepare_data(CONFIG)

________________________________________________________________________________________________________________________________
                                                         Preparing Data                                                         

 - SSL dataset:           BraTS
 - SSL Train samples:       1251
 - Survival Train samples:   482
 - Survival Val samples:      60
 - Survival Test samples:     61


## Step 1: SSL Pretraining

Contrastive SSL on BraTS tensors. Trains from scratch (no ViT pretraining step).

In [ ]:
print_separator("Starting SSL Pretraining", sep_type='SUPER')
ssl_checkpoint_path = train_stage_ssl(
    CONFIG,
    ssl_train_loader,
)

## Step 2: Survival Training

Loads the best SSL checkpoint (selected by val_loss) into the multimodal survival model.

In [ ]:
print_separator("Starting Survival Training", sep_type='SUPER')
survival_module = train_stage_survival(
    CONFIG, ssl_checkpoint_path,
    survival_dm, train_loader,
    val_loader, test_loader
)

## Step 3: Testing

Separate test step on the test loader.

In [ ]:
print_separator("Step 3: Testing", sep_type='SUPER')
results = test_model(CONFIG, survival_module, test_loader)

In [ ]:
results['predictions']

---

In [8]:
CHECKPOINTS = [
    # (path, use_radiomics, label)
    # ('../models/brats-All_masks-No_Radiomics_2026-05-25-01-44-34/ssl_pretraining_best.ckpt', False, 'brats-All_masks-No_Radiomics'),
    # # ('../models/brats-All_masks-Radiomics_2026-05-25-02-39-42/ssl_pretraining_best.ckpt', True, 'brats-All_masks-Radiomics'),
    ('../models/brats-No_masks-No_Radiomics_2026-05-24-21-58-24/ssl_pretraining_best.ckpt', False, 'brats-No_masks-No_Radiomics'),
    ('../models/brats-No_masks-Radiomics_2026-05-24-23-53-23/ssl_pretraining_best.ckpt', True, 'brats-No_masks-Radiomics'),
    # ('../models/brats-Train_masks-No_Radiomics_2026-05-24-22-56-10/ssl_pretraining_best.ckpt', False, 'brats-Train_masks-No_Radiomics'),
    # # ('../models/brats-Train_masks-Radiomics_2026-05-25-00-49-00/ssl_pretraining_best.ckpt', True, 'brats-Train_masks-Radiomics'),
    # # ('../models/Upenn-All_masks-No_Radiomics_2026-05-24-21-00-42/ssl_pretraining_best.ckpt', False, 'Upenn-All_masks-No_Radiomics'),
    # # ('../models/Upenn-All_masks-Radiomics_2026-05-24-21-29-45/ssl_pretraining_best.ckpt', True, 'Upenn-All_masks-Radiomics'),
    ('../models/Upenn-No_masks-No_Radiomics_2026-05-24-19-02-37/ssl_pretraining_best.ckpt', False, 'Upenn-No_masks-No_Radiomics'),
    ('../models/Upenn-No_masks-Radiomics_2026-05-24-20-01-26/ssl_pretraining_best.ckpt', True, 'Upenn-No_masks-Radiomics'),
    # # ('../models/Upenn-Train_masks-No_Radiomics_2026-05-24-19-31-44/ssl_pretraining_best.ckpt', False, 'Upenn-Train_masks-No_Radiomics'),
    # # ('../models/Upenn-Train_masks-Radiomics_2026-05-24-20-31-53/ssl_pretraining_best.ckpt', True, 'Upenn-Train_masks-Radiomics'),
]


In [9]:
import os
from maikol_utils.file_utils import save_json

for ssl_checkpoint_path, use_radiomics, label in CHECKPOINTS:
    
    CONFIG = Configuration(
        survival_epochs=5,
        masked_train='Train_masks' in label or 'All_masks' in label,
        masked_test=True,  # Assuming test masking follows training masking
        pos_embed="1d",
        use_radiomics=use_radiomics,
        dynamic_dropout=True,
        ssl_dataset="brats" if 'brats' in label else "upenn",
        exp_name=f"{label}-test",
    )


    ssl_train_loader, survival_dm, train_loader, val_loader, test_loader = prepare_data(CONFIG)

    print_separator(f"Starting Survival Training for {label}", sep_type='SUPER')
    survival_module = train_stage_survival(
        CONFIG, ssl_checkpoint_path,
        survival_dm, train_loader,
        val_loader, test_loader
    )

    print_separator("Step 3: Testing", sep_type='SUPER')
    results = test_model(CONFIG, survival_module, test_loader)
    save_json(os.path.join(CONFIG.MODELS_PATH, "results.json"), results)

Configuration initialized
 - seed: 42
 - masked_train: False
 - masked_test: True
 - dataset: brats
 - epochs: 30 (SSL), 5 (Survival)
 - use_radiomics: False
 - dynamic_dropout: True
 - batch_size: 32 (SSL), 16 (Survival)
 - SSL dataset:           BraTS
 - SSL Train samples:       1251


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                        | Params | Mode  | FLOPs
--------------------------------------------------------------------------
0 | model     | MultimodalSurvivalPredictor | 10.0 M | train | 0    
1 | criterion | CrossEntropyLoss            | 0      | train | 0    
--------------------------------------------------------------------------
10.0 M    Trainable params
0         Non-trainable params
10.0 M    Total params
40.192    Total estimated model params size (MB)
97        Modules in train mode
0         Modules in eval mode
0         Total Flops


 - Survival Train samples:   482
 - Survival Val samples:      60
 - Survival Test samples:     61
                                   Starting Survival Training for brats-No_masks-No_Radiomics                                   

[pretrained encoder] missing=0  unexpected=0
 - [Survival] device: gpu:0 (NVIDIA GeForce RTX 5070)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

`Trainer.fit` stopped: `max_epochs=5` reached.
Restoring states from the checkpoint path at /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/Upenn-No_masks-Radiomics-test_2026-05-25-16-31-15/survival_checkpoint_best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/Upenn-No_masks-Radiomics-test_2026-05-25-16-31-15/survival_checkpoint_best.ckpt
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.5901639461517334
        test_bacc           0.5860070586204529
         test_f1            0.5761128664016724
        test_loss           0.6705703735351562
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 - Best Survival checkpoint: /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/Upenn-No_masks-Radiomics-test_2026-05-25-16-31-15/survival_checkpoint_best.ckpt
                                                        Step 3: Testing                                                         



Testing: 100%|██████████| 4/4 [00:01<00:00,  2.28it/s]

 - Test F1 Score:  0.5868
 - Test Precision: 0.5915
 - Test Recall:    0.5902
 - Test Accuracy:  0.5902
Saving output at ../models/Upenn-No_masks-Radiomics-test_2026-05-25-16-31-15/results.json...


In [7]:
import os
from maikol_utils.file_utils import save_json

for ssl_checkpoint_path, use_radiomics, label in CHECKPOINTS:
    
    CONFIG = Configuration(
        survival_epochs=5,
        masked_train='Train_masks' in label or 'All_masks' in label,
        masked_test='All_masks' in label,  # Assuming test masking follows training masking
        pos_embed="1d",
        use_radiomics=use_radiomics,
        dynamic_dropout=True,
        ssl_dataset="brats" if 'brats' in label else "upenn",
        exp_name=label,
    )


    ssl_train_loader, survival_dm, train_loader, val_loader, test_loader = prepare_data(CONFIG)

    print_separator(f"Starting Survival Training for {label}", sep_type='SUPER')
    survival_module = train_stage_survival(
        CONFIG, ssl_checkpoint_path,
        survival_dm, train_loader,
        val_loader, test_loader
    )

    print_separator("Step 3: Testing", sep_type='SUPER')
    results = test_model(CONFIG, survival_module, test_loader)
    save_json(os.path.join(CONFIG.MODELS_PATH, "results.json"), results)

Configuration initialized
 - seed: 42
 - masked_train: True
 - masked_test: True
 - dataset: brats
 - epochs: 30 (SSL), 5 (Survival)
 - use_radiomics: False
 - dynamic_dropout: True
 - batch_size: 32 (SSL), 16 (Survival)
 - SSL dataset:           BraTS
 - SSL Train samples:       1251


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                        | Params | Mode  | FLOPs
--------------------------------------------------------------------------
0 | model     | MultimodalSurvivalPredictor | 10.0 M | train | 0    
1 | criterion | CrossEntropyLoss            | 0      | train | 0    
--------------------------------------------------------------------------
10.0 M    Trainable params
0         Non-trainable params
10.0 M    Total params
40.192    Total estimated model params size (MB)
97        Modules in train mode
0         Modules in eval mode
0         Total Flops


 - Survival Train samples:   482
 - Survival Val samples:      60
 - Survival Test samples:     61
                                  Starting Survival Training for brats-All_masks-No_Radiomics                                   

[pretrained encoder] missing=0  unexpected=0
 - [Survival] device: gpu:0 (NVIDIA GeForce RTX 5070)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

`Trainer.fit` stopped: `max_epochs=5` reached.
Restoring states from the checkpoint path at /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/Upenn-Train_masks-Radiomics_2026-05-25-16-17-37/survival_checkpoint_best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/Upenn-Train_masks-Radiomics_2026-05-25-16-17-37/survival_checkpoint_best.ckpt
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.5737704634666443
        test_bacc            0.584133505821228
         test_f1            0.5605247020721436
        test_loss           0.6915757060050964
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 - Best Survival checkpoint: /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/Upenn-Train_masks-Radiomics_2026-05-25-16-17-37/survival_checkpoint_best.ckpt
                                                        Step 3: Testing                                                         



Testing: 100%|██████████| 4/4 [00:02<00:00,  1.66it/s]

 - Test F1 Score:  0.5653
 - Test Precision: 0.5773
 - Test Recall:    0.5738
 - Test Accuracy:  0.5738
Saving output at ../models/Upenn-Train_masks-Radiomics_2026-05-25-16-17-37/results.json...
